# Phase 2 Parser & Validation Pipeline Verification

This notebook loads the published Phase 2 data-quality reports, verifies the physical line outcome invariant, inspects the anomaly code distributions, displays representative quarantined examples for each error code, and presents the audit and cohort reconciliations.


In [1]:
import json
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

DATA_DIR = Path('../data')
INTERIM_DIR = DATA_DIR / 'interim'
REPORTS_DIR = Path('../reports/data_quality')

# Fallback if executing from root
if not REPORTS_DIR.exists() and Path('reports/data_quality').exists():
    INTERIM_DIR = Path('data/interim')
    REPORTS_DIR = Path('reports/data_quality')

print(f'Interim dir: {INTERIM_DIR.resolve()}')
print(f'Reports dir: {REPORTS_DIR.resolve()}')


Interim dir: C:\Users\prane\.vscode\extensions\ms-vscode.cmake-tools-1.21.36\ml project\PARKINSONS-DISEASE-PREDICTOR\data\interim
Reports dir: C:\Users\prane\.vscode\extensions\ms-vscode.cmake-tools-1.21.36\ml project\PARKINSONS-DISEASE-PREDICTOR\reports\data_quality


In [2]:
with open(INTERIM_DIR / 'run_manifest.json', 'r', encoding='utf-8') as f:
    manifest = json.load(f)

with open(REPORTS_DIR / 'summary.json', 'r', encoding='utf-8') as f:
    summary = json.load(f)

print('=== EXECUTION MANIFEST ===')
for k, v in manifest.items():
    print(f'{k}: {v}')

print('\n=== RUN SUMMARY ===')
for k, v in summary.items():
    print(f'{k}: {v}')


=== EXECUTION MANIFEST ===
git_commit: 26aaba43072724e474117be253a3c929180503f1
git_dirty: False
python_version: 3.13.7
os_system: Windows
package_versions: {'pandas': '3.0.1', 'numpy': '2.3.5', 'pyarrow': '25.0.1'}
config_hash_sha256: e2f5cfe492c6a1072232a3cc09e77abe85dd5a528fad6b1a79fd6773f26e9bbd
raw_hashes_verified: True
total_raw_files_verified: 2812
files_processed: 2165

=== RUN SUMMARY ===
files_scanned: 2165
total_lines: 40524886
n_accepted: 39745054
n_warned: 314625
n_quarantined: 465207
n_blank: 0
unique_log_participants: 754
unique_metadata_participants: 646
participants_in_both_sources: 609
eligible_cohort_count: 597
raw_hashes_verified: True
git_commit: 26aaba43072724e474117be253a3c929180503f1
git_dirty: False


In [3]:
tot_lines = summary['total_lines']
n_acc = summary['n_accepted']
n_warn = summary['n_warned']
n_quar = summary['n_quarantined']
n_blk = summary['n_blank']

invariant_holds = (tot_lines == n_acc + n_warn + n_quar + n_blk)
print(f'Physical lines scanned : {tot_lines:,}')
print(f'  Accepted (clean)     : {n_acc:,} ({n_acc / tot_lines * 100:.2f}%)')
print(f'  Accepted (w/ warning): {n_warn:,} ({n_warn / tot_lines * 100:.2f}%)')
print(f'  Quarantined          : {n_quar:,} ({n_quar / tot_lines * 100:.2f}%)')
print(f'  Blank lines          : {n_blk:,} ({n_blk / tot_lines * 100:.2f}%)')
print(f'Invariant holds        : {invariant_holds}')
assert invariant_holds, 'Line-outcome invariant violated!'


Physical lines scanned : 40,524,886
  Accepted (clean)     : 39,745,054 (98.08%)
  Accepted (w/ warning): 314,625 (0.78%)
  Quarantined          : 465,207 (1.15%)
  Blank lines          : 0 (0.00%)
Invariant holds        : True


In [4]:
codes_df = pd.read_csv(REPORTS_DIR / 'code_counts.csv')
display(codes_df)


,code,code_type,total_occurrences,unique_lines_affected,accepted_lines_count,quarantined_lines_count,description
0,E01_FIELD_COUNT,ERROR,0,0,0,0,Number of fields differs from canonical shape
1,E02_BAD_ENCODING,ERROR,0,0,0,0,Source-line bytes are not valid UTF-8
2,E03_BAD_USERKEY,ERROR,387,387,0,387,Key is not exactly 10 uppercase letters or digits
3,E04_USERKEY_MISMATCH,ERROR,387,387,0,387,Record key differs from filename key
4,E05_BAD_DATE,ERROR,291,291,0,291,Date is not 6 digits or not a valid calendar date
5,E06_BAD_TIME,ERROR,196341,196341,0,196341,Timestamp is not HH:MM:SS.SSS or out of range
6,E07_BAD_HAND,ERROR,1043,1043,0,1043,"Hand is outside allowed set {L, R, S}"
7,E08_BAD_HOLD,ERROR,463687,463687,0,463687,Hold time is not a plain decimal or is negative
8,E09_BAD_DIRECTION,ERROR,1489,1489,0,1489,Direction is outside the 9 allowed transitions
9,E10_BAD_LATENCY,ERROR,464119,464119,0,464119,Latency is not a plain decimal or is negative


In [5]:
quarantine_dir = INTERIM_DIR / 'quarantine'
quar_files = sorted(list(quarantine_dir.glob('*.parquet')))

examples_by_code = {}
target_codes = [
    'E01_FIELD_COUNT', 'E02_BAD_ENCODING', 'E03_BAD_USERKEY', 'E04_USERKEY_MISMATCH',
    'E05_BAD_DATE', 'E06_BAD_TIME', 'E07_BAD_HAND', 'E08_BAD_HOLD',
    'E09_BAD_DIRECTION', 'E10_BAD_LATENCY', 'E11_BAD_FLIGHT'
]

for code in target_codes:
    examples_by_code[code] = []

for qf in quar_files:
    tbl = pq.read_table(qf)
    df_q = tbl.to_pandas()
    for _, row in df_q.iterrows():
        reasons = row['reason_codes'].split(',')
        for r in reasons:
            if r in examples_by_code and len(examples_by_code[r]) < 5:
                examples_by_code[r].append({
                    'source_file': row['source_file'],
                    'line_no': row['line_no'],
                    'code': r,
                    'raw_preview': row['raw_line_display'][:80],
                })
    if all(len(v) >= 5 for k, v in examples_by_code.items() if k not in ['E01_FIELD_COUNT', 'E02_BAD_ENCODING']):
        break

print('=== QUARANTINE EXAMPLES PER ERROR CODE ===\n')
for code, ex_list in examples_by_code.items():
    print(f'Code: {code} (Found {len(ex_list)} examples)')
    for ex in ex_list[:5]:
        print(f"  [{ex['source_file']}:L{ex['line_no']}] {repr(ex['raw_preview'])}")
    print()


=== QUARANTINE EXAMPLES PER ERROR CODE ===

Code: E01_FIELD_COUNT (Found 0 examples)

Code: E02_BAD_ENCODING (Found 0 examples)

Code: E03_BAD_USERKEY (Found 5 examples)
  [0EA27ICBLF_1607.txt:L2651] '0EA27ICB0EA27ICBLF\t160723\t17:35:20.516\tL\t0199.2\tLL\t0453.1\t0281.3\t'
  [0EA27ICBLF_1607.txt:L94426] '00EA27ICBLF\t160725\t23:51:23.453\tL\t0101.6\tLL\t0320.3\t0171.9\t'
  [0QAZFRHQHW_1707.txt:L345] '0QAZFRHQHW0QAZFRHQHW\t170704\t19:07:50.320\tR\t0117.2\tLR\t0335.9\t0140.6\t'
  [2JTCBKUP8T_1704.txt:L10710] 'KUP8T\t170410\t23:23:47.359\tL\t0132.8\tRL\t0468.8\t0390.6\t'
  [2JTCBKUP8T_1704.txt:L11496] '\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00'

Code: E04_USERKEY_MISMATCH (Found 5 examples)
  [0EA27ICBLF_1607.txt

In [6]:
cohort_df = pd.read_csv(REPORTS_DIR / 'cohort_reconciliation.csv')
print('=== COHORT RECONCILIATION ===')
display(cohort_df)


=== COHORT RECONCILIATION ===


,metric,phase1_baseline,phase2_value,difference,explanation
0,total_unique_participants,791,791,0,754 with logs + 37 with metadata only
1,participants_with_logs,754,754,0,Confirmed via monthly log file prefixes
2,participants_with_metadata,646,646,0,Confirmed via User_*.txt files
3,participants_in_both_sources,609,609,0,Intersection of log keys and metadata keys
4,log_participants_without_metadata,145,145,0,Participants present in logs but missing metadata
5,metadata_participants_without_logs,37,37,0,Participants present in metadata but missing logs
6,eligible_participants,606,597,-9,"Participants with logs, metadata, valid label,..."
7,joint_participant_excluded:DRQKANH3Q0,0,1,1,Participant in both logs and metadata but excl...
8,joint_participant_excluded:EIFBUG1US3,0,1,1,Participant in both logs and metadata but excl...
9,joint_participant_excluded:FXD2HCNIPJ,0,1,1,Participant in both logs and metadata but excl...


In [7]:
audit_df = pd.read_csv(REPORTS_DIR / 'audit_reconciliation.csv')
print('=== AUDIT RECONCILIATION ===')
display(audit_df)


=== AUDIT RECONCILIATION ===


,metric,phase1_baseline,phase2_value,difference,explanation_status
0,monthly_log_files_scanned,2165,2165,0,Matches exact count of KEY_YYMM.txt logs (Arch...
1,nonblank_physical_lines,40524886,40524886,0,Exact match across all physical lines scanned
2,blank_lines,0,0,0,Matches baseline zero blank physical lines
3,field_count_not_eight (strict tab),4,0,-4,Exact match: 4 lines with 7 fields due to miss...
4,participant_id_mismatch (tab),387,387,0,Exact match with strict tab ID check (384 + 3 ...
5,invalid_date_rows,291,291,0,Exact match on date format and calendar valida...
6,invalid_time_rows,196341,196341,0,Exact match on timestamp format and range vali...
7,invalid_hand_rows,1043,1043,0,"Exact match on allowed hands {L, R, S}"
8,timing_parse_error_rows,464803,464119,-684,"Exact match: 464,803 lines with comma decimal ..."
9,backward_timestamp_transitions,44748,44748,0,Phase 2 strict full datetime comparison across...


In [8]:
hash_df = pd.read_csv(REPORTS_DIR / 'raw_files_sha256.csv')
all_matched = hash_df['is_identical'].all()
print(f'Total raw files checked: {len(hash_df):,}')
print(f'All raw files untampered: {all_matched}')
assert all_matched, 'Raw file integrity failure!'


Total raw files checked: 2,812
All raw files untampered: True
